# VQ-VAE (Neural Discrete Representation Learning) — Roll No. 24CS2015 (DL-081)
**Plan:** (1) build VQ layer + autoencoder, (2) train it on CIFAR-10, (3) ablations: EMA vs gradient codebook, codebook size K, (4) study codebook collapse, (5) train a PixelCNN prior over the codes and sample new images.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all.

> Read the markdown above every cell. You must be able to explain each line in the viva.

## 0. Setup
Imports, a fixed random seed (so results can be repeated) and the GPU/CPU choice.

In [ ]:
import random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = 10          # lower (e.g. 2) for a quick test, raise (e.g. 20) for final results
BATCH = 128

## 1. Data (CIFAR-10)
Images are scaled to [-1, 1]. `data_var` is the pixel variance; we divide the reconstruction error by it so the loss is on a standard scale (as in the paper's reference setups).

In [ ]:
tf = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_ds = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
test_ds  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, drop_last=True)
test_dl  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)
data_var = float(np.var(train_ds.data / 255.0) / 0.25)
print("data_var:", data_var)

## 2. The core idea: Vector Quantiser (YOUR contribution — know this cold)
- Encoder outputs a vector `z` at every spatial position.
- We have a **codebook** of `K` vectors of size `D`. Replace each `z` by its **nearest** codebook vector `e_k` (this is the "discrete" part).
- `argmin` has no gradient, so we use the **straight-through estimator**: `z_q = z + (e_k - z).detach()` → forward value is `e_k`, backward gradient flows to `z` as if nothing happened.
- Loss terms from the paper: `L = ||x - x̂||² + ||sg[z] - e||² + β||z - sg[e]||²` (sg = stop-gradient).
  - 2nd term (codebook loss) moves codewords toward encoder outputs.
  - 3rd term (commitment loss) keeps the encoder from drifting; β = 0.25.
- **EMA variant** (paper appendix): instead of the 2nd loss term, update the codebook by an exponential moving average of the encoder vectors assigned to each code.
- **Perplexity** = exp(entropy of code usage). Near K = all codes used; near 1 = collapse.

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=512, dim=64, beta=0.25, ema=True, decay=0.99, eps=1e-5):
        super().__init__()
        self.K, self.D, self.beta, self.ema, self.decay, self.eps = num_embeddings, dim, beta, ema, decay, eps
        self.embedding = nn.Embedding(self.K, self.D)
        self.embedding.weight.data.uniform_(-1.0 / self.K, 1.0 / self.K)
        if ema:
            self.embedding.weight.requires_grad = False          # codebook is NOT trained by gradients
            self.register_buffer("ema_count", torch.zeros(self.K))
            self.register_buffer("ema_weight", self.embedding.weight.data.clone())

    def forward(self, z):                                        # z: (B, D, H, W)
        B, D, H, W = z.shape
        z = z.permute(0, 2, 3, 1).contiguous()                   # (B, H, W, D)
        flat = z.view(-1, D)                                     # (B*H*W, D)
        w = self.embedding.weight                                # (K, D)

        # squared distance to every codeword: |z|^2 - 2 z.e + |e|^2
        dist = flat.pow(2).sum(1, keepdim=True) - 2 * flat @ w.t() + w.pow(2).sum(1)
        idx = dist.argmin(1)                                     # nearest codeword per position
        onehot = F.one_hot(idx, self.K).float()
        zq = (onehot @ w).view_as(z)                             # quantised vectors

        if self.training and self.ema:                           # EMA codebook update (no gradients)
            with torch.no_grad():
                n = onehot.sum(0)
                self.ema_count.mul_(self.decay).add_(n, alpha=1 - self.decay)
                total = self.ema_count.sum()
                self.ema_count.copy_((self.ema_count + self.eps) / (total + self.K * self.eps) * total)
                self.ema_weight.mul_(self.decay).add_(onehot.t() @ flat, alpha=1 - self.decay)
                self.embedding.weight.data.copy_(self.ema_weight / self.ema_count.unsqueeze(1))

        commit = F.mse_loss(zq.detach(), z)                      # ||z - sg[e]||^2
        if self.ema:
            vq_loss = self.beta * commit
        else:
            codebook = F.mse_loss(zq, z.detach())                # ||sg[z] - e||^2
            vq_loss = codebook + self.beta * commit

        zq = z + (zq - z).detach()                               # straight-through estimator
        avg = onehot.mean(0)
        perplexity = torch.exp(-(avg * torch.log(avg + 1e-10)).sum())
        return zq.permute(0, 3, 1, 2).contiguous(), vq_loss, perplexity, idx.view(B, H, W)

## 3. Encoder, Decoder, full VQ-VAE
Image 32×32 → encoder (two stride-2 convs) → 8×8 grid of vectors → quantise → decoder → 32×32 image.
So every image is stored as just **64 integers** (8×8 code indices).

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, ch):
        super().__init__()
        self.f = nn.Sequential(nn.ReLU(), nn.Conv2d(c, ch, 3, 1, 1), nn.ReLU(), nn.Conv2d(ch, c, 1))
    def forward(self, x): return x + self.f(x)

class Encoder(nn.Module):
    def __init__(self, hid=128, D=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, hid, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hid, hid, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hid, hid, 3, 1, 1),
            ResBlock(hid, 32), ResBlock(hid, 32), nn.ReLU(),
            nn.Conv2d(hid, D, 1))
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self, hid=128, D=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(D, hid, 3, 1, 1),
            ResBlock(hid, 32), ResBlock(hid, 32), nn.ReLU(),
            nn.ConvTranspose2d(hid, hid, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(hid, 3, 4, 2, 1))
    def forward(self, z): return self.net(z)

class VQVAE(nn.Module):
    def __init__(self, K=512, D=64, hid=128, beta=0.25, ema=True):
        super().__init__()
        self.enc, self.vq, self.dec = Encoder(hid, D), VectorQuantizer(K, D, beta, ema), Decoder(hid, D)
    def forward(self, x):
        zq, vq_loss, perp, idx = self.vq(self.enc(x))
        return self.dec(zq), vq_loss, perp, idx

## 4. Training + evaluation helpers
We log, per epoch: reconstruction loss, perplexity, and **how many codes are actually used** (to study collapse).

In [ ]:
@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    mse, n = 0.0, 0
    counts = torch.zeros(model.vq.K, device=device)
    for x, _ in dl:
        x = x.to(device)
        xr, _, _, idx = model(x)
        mse += F.mse_loss(xr, x, reduction="sum").item(); n += x.numel()
        counts += torch.bincount(idx.flatten(), minlength=model.vq.K).float()
    p = counts / counts.sum()
    perp = torch.exp(-(p * torch.log(p + 1e-10)).sum()).item()
    return dict(test_mse=mse / n, test_perplexity=perp, codes_used=int((counts > 0).sum().item()), counts=counts.cpu().numpy())

def train_vqvae(K=512, D=64, ema=True, beta=0.25, epochs=EPOCHS, lr=2e-4, tag=""):
    torch.manual_seed(SEED)
    model = VQVAE(K, D, beta=beta, ema=ema).to(device)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    hist = dict(recon=[], perp=[])
    for ep in range(epochs):
        model.train(); r_sum = p_sum = 0.0; steps = 0
        for x, _ in train_dl:
            x = x.to(device)
            xr, vq_loss, perp, _ = model(x)
            recon = F.mse_loss(xr, x) / data_var
            loss = recon + vq_loss
            opt.zero_grad(); loss.backward(); opt.step()
            r_sum += recon.item(); p_sum += perp.item(); steps += 1
        hist["recon"].append(r_sum / steps); hist["perp"].append(p_sum / steps)
        print(f"[{tag}] epoch {ep+1}/{epochs}  recon {hist['recon'][-1]:.4f}  perplexity {hist['perp'][-1]:.1f}")
    res = evaluate(model, test_dl)
    print(f"[{tag}] TEST mse {res['test_mse']:.5f} | perplexity {res['test_perplexity']:.1f} | codes used {res['codes_used']}/{K}")
    return model, hist, res

## 5. Main model (reproduction result)
K = 512 codes, EMA codebook updates — the paper's recommended setup.

In [ ]:
main_model, main_hist, main_res = train_vqvae(K=512, ema=True, tag="K512-EMA")

### Reconstructions (top = original, bottom = reconstruction)

In [ ]:
@torch.no_grad()
def show_recon(model, n=10):
    model.eval()
    x, _ = next(iter(test_dl)); x = x[:n].to(device)
    xr = model(x)[0].clamp(-1, 1)
    grid = torch.cat([x, xr]).cpu() * 0.5 + 0.5
    g = torchvision.utils.make_grid(grid, nrow=n, padding=1)
    plt.figure(figsize=(14, 3)); plt.imshow(g.permute(1, 2, 0)); plt.axis("off"); plt.show()
show_recon(main_model)

## 6. Ablations (this is where the marks are)
1. **EMA vs gradient codebook** (no EMA) — same K.
2. **Codebook size K** = 64 / 512 / 2048 — look at perplexity & codes used.
Each run takes a few minutes on a T4. Reduce `EPOCHS` if short on time.

In [ ]:
runs = {"K512-EMA": (main_model, main_hist, main_res)}
runs["K512-noEMA"] = train_vqvae(K=512,  ema=False, tag="K512-noEMA")
runs["K64-EMA"]    = train_vqvae(K=64,   ema=True,  tag="K64-EMA")
runs["K2048-EMA"]  = train_vqvae(K=2048, ema=True,  tag="K2048-EMA")

### Results table + plots (copy these into your report)

In [ ]:
print(f"{'run':14s} {'test MSE':>10s} {'perplexity':>11s} {'codes used':>11s}")
for name, (_, _, r) in runs.items():
    print(f"{name:14s} {r['test_mse']:10.5f} {r['test_perplexity']:11.1f} {r['codes_used']:>7d}/{runs[name][0].vq.K}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, h, _) in runs.items():
    ax[0].plot(h["recon"], label=name); ax[1].plot(h["perp"], label=name)
ax[0].set_title("Train reconstruction loss (normalised)"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("Codebook perplexity (higher = more codes used)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("ablation_curves.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, len(runs), figsize=(4 * len(runs), 3))
for a, (name, (_, _, r)) in zip(ax, runs.items()):
    a.bar(range(len(r["counts"])), np.sort(r["counts"])[::-1]); a.set_title(name + " usage (sorted)")
plt.tight_layout(); plt.savefig("codebook_usage.png", dpi=150); plt.show()

## 7. Prior over codes (PixelCNN) → generate NEW images
The VQ-VAE can only compress/reconstruct. To **generate**, we learn p(code grid) with an autoregressive model: predict each code from the codes before it (raster order). Then sample a code grid and decode it.
`MaskedConv2d` hides "future" positions so the model can't cheat.

In [ ]:
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *a, **k):
        super().__init__(*a, **k)
        self.register_buffer("mask", torch.ones_like(self.weight))
        _, _, kh, kw = self.weight.shape
        self.mask[:, :, kh // 2, kw // 2 + (mask_type == "B"):] = 0   # right of centre (A also hides centre)
        self.mask[:, :, kh // 2 + 1:] = 0                              # rows below
    def forward(self, x):
        self.weight.data *= self.mask
        return super().forward(x)

class PixelCNNPrior(nn.Module):
    def __init__(self, K, ch=128, layers=8):
        super().__init__()
        self.K = K
        self.emb = nn.Embedding(K, ch)
        self.first = MaskedConv2d("A", ch, ch, 7, padding=3)
        self.blocks = nn.ModuleList([MaskedConv2d("B", ch, ch, 3, padding=1) for _ in range(layers)])
        self.out = nn.Sequential(nn.ReLU(), nn.Conv2d(ch, ch, 1), nn.ReLU(), nn.Conv2d(ch, K, 1))
    def forward(self, idx):                                   # idx: (B, H, W) long
        h = self.first(self.emb(idx).permute(0, 3, 1, 2))
        for b in self.blocks: h = h + b(F.relu(h))
        return self.out(h)                                    # (B, K, H, W) logits

@torch.no_grad()
def encode_dataset(model, dl):
    model.eval(); out = []
    for x, _ in dl: out.append(model(x.to(device))[3].cpu())
    return torch.cat(out)

def train_prior(model, epochs=15, lr=1e-3):
    codes_tr, codes_te = encode_dataset(model, train_dl), encode_dataset(model, test_dl)
    K = model.vq.K; prior = PixelCNNPrior(K).to(device)
    opt = torch.optim.Adam(prior.parameters(), lr=lr)
    for ep in range(epochs):
        prior.train(); perm = torch.randperm(len(codes_tr)); tot = 0; nb = 0
        for i in range(0, len(perm) - 128, 128):
            b = codes_tr[perm[i:i + 128]].to(device)
            loss = F.cross_entropy(prior(b), b)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        prior.eval()
        with torch.no_grad():
            te = np.mean([F.cross_entropy(prior(codes_te[i:i + 256].to(device)), codes_te[i:i + 256].to(device)).item()
                          for i in range(0, len(codes_te) - 256, 256)])
        print(f"prior epoch {ep+1}/{epochs}  train NLL {tot/nb:.3f}  test NLL {te:.3f}")
    return prior

@torch.no_grad()
def sample_images(model, prior, n=32, H=8, W=8, temperature=1.0):
    prior.eval(); model.eval()
    codes = torch.zeros(n, H, W, dtype=torch.long, device=device)
    for i in range(H):
        for j in range(W):
            logits = prior(codes)[:, :, i, j] / temperature
            codes[:, i, j] = torch.multinomial(F.softmax(logits, -1), 1).squeeze(1)
    zq = model.vq.embedding(codes).permute(0, 3, 1, 2)
    return model.dec(zq).clamp(-1, 1).cpu() * 0.5 + 0.5

prior = train_prior(main_model)
imgs = sample_images(main_model, prior)
g = torchvision.utils.make_grid(imgs, nrow=8, padding=1)
plt.figure(figsize=(10, 5)); plt.imshow(g.permute(1, 2, 0)); plt.axis("off")
plt.savefig("generated_samples.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Write down for your report
- Reproduction: paper reports CIFAR-10 reconstructions/samples (Sec. 4); we trained a **much smaller** model for few epochs — say this honestly.
- Ablation findings (fill from your table): does EMA give higher perplexity / lower MSE? What happens with K=64 vs 2048?
- Failure cases: blurry reconstructions, blurry/odd samples from the prior, any low-perplexity (collapsed) run.
- Cite: van den Oord et al., NeurIPS 2017 (arXiv 1711.00937) and any GitHub repo you looked at.